# Manipulación de datos con Pandas — datos.gob.do

**Estudiante:** Brailin Leonardo Méndez Guillermo  
**Objetivo:** Limpiar y filtrar resultados oficiales de Pruebas Nacionales por centro educativo, publicados por MINERD en datos.gob.do.


Fuente oficial: conjunto **Resultados de Pruebas Nacionales por Centro Educativo, 2016-2024**, catálogo de datos.gob.do.

In [ ]:
import requests, pandas as pd, io, re

API = "https://datos.gob.do/api/3/action"

def csv_resource_from_package(package_id):
    meta = requests.get(f"{API}/package_show", params={"id": package_id}, timeout=30).json()
    assert meta.get("success"), meta
    resources = meta["result"]["resources"]
    csvs = [r for r in resources if (r.get("format") or "").upper() == "CSV"]
    if not csvs:
        raise RuntimeError("El conjunto no ofrece un recurso CSV")
    return csvs[-1]["url"], meta["result"]["title"]

def read_csv_robust(url):
    raw = requests.get(url, timeout=60).content
    for enc in ("utf-8-sig", "latin-1", "cp1252"):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc, sep=None, engine="python")
        except UnicodeDecodeError:
            pass
    raise RuntimeError("No fue posible detectar la codificación")


In [ ]:
PACKAGE_ID = "c99df9fc-6d99-4cb3-8f2b-81d5753d51b7"
url_csv, titulo = csv_resource_from_package(PACKAGE_ID)
df = read_csv_robust(url_csv)
print(titulo, df.shape)
print(url_csv)
df.head()


In [ ]:
# Normalización y limpieza
df.columns = [re.sub(r'[^0-9a-záéíóúñ]+', '_', str(c).strip().lower()).strip('_') for c in df.columns]
df = df.drop_duplicates().copy()
for c in df.columns:
    if df[c].dtype == 'object':
        df[c] = df[c].astype(str).str.strip()
        posible = pd.to_numeric(df[c].str.replace(',', '', regex=False), errors='coerce')
        if posible.notna().mean() >= .75:
            df[c] = posible
print("Filas duplicadas eliminadas y columnas normalizadas.")
display(df.isna().sum().sort_values(ascending=False).head(12))


In [ ]:
# Identificación flexible de columnas
def buscar_columna(*terminos):
    for c in df.columns:
        if all(t in c for t in terminos):
            return c
    return None

c_regional = buscar_columna('regional')
c_estudiantes = buscar_columna('cantidad', 'estudiante') or buscar_columna('estudiante')
c_promovidos = buscar_columna('promov')
c_aplazados = buscar_columna('aplaz')
print(c_regional, c_estudiantes, c_promovidos, c_aplazados)


In [ ]:
# Manipulaciones y condicionales
trabajo = df.copy()
if c_estudiantes and c_promovidos:
    trabajo['tasa_promocion'] = (100 * trabajo[c_promovidos] / trabajo[c_estudiantes].replace(0, pd.NA)).round(2)
    alto_volumen = trabajo[trabajo[c_estudiantes] >= trabajo[c_estudiantes].quantile(.75)]
    alta_promocion = trabajo[trabajo['tasa_promocion'] >= 90]
    print("Centros en cuartil superior de estudiantes:", len(alto_volumen))
    print("Centros con promoción >= 90%:", len(alta_promocion))
    display(alta_promocion.sort_values('tasa_promocion', ascending=False).head(10))

if c_regional and c_estudiantes:
    resumen = (trabajo.groupby(c_regional, dropna=False)[c_estudiantes]
               .agg(['count','sum','mean']).round(2).sort_values('sum', ascending=False))
    display(resumen.head(15))


In [ ]:
import matplotlib.pyplot as plt
if c_regional and c_estudiantes:
    resumen['sum'].head(10).sort_values().plot(kind='barh', figsize=(10,5), title='Estudiantes por regional (10 principales)')
    plt.xlabel('Cantidad de estudiantes'); plt.tight_layout(); plt.show()
if 'tasa_promocion' in trabajo:
    trabajo['tasa_promocion'].dropna().plot(kind='hist', bins=20, figsize=(9,4), title='Distribución de la tasa de promoción')
    plt.xlabel('% de promoción'); plt.grid(alpha=.2); plt.show()


### Conclusión
La práctica demuestra limpieza, conversión de tipos, eliminación de duplicados, filtros condicionales, creación de una métrica derivada, agrupaciones y visualización. Los filtros usados —volumen de estudiantes y promoción— son pertinentes para el conjunto oficial seleccionado.